# Generate Combined ERA5 and ECMWF Forecast Dataset

This notebook fetches, merges, and validates historical weather reanalysis (ERA5) and future forecasts (ECMWF IFS) from the Open-Meteo APIs for Jerukagung coordinate. It outputs a validated CSV dataset to replace `cuaca_gabungan_jerukagung.csv`.

In [1]:
# 1. Setup Client and Imports
import sys
import subprocess
import os

# Auto-install external dependencies if missing (especially useful in Kaggle env)
for pkg in ["openmeteo-requests", "requests-cache", "retry-requests"]:
    module_name = pkg.replace("-", "_")
    try:
        __import__(module_name)
    except ImportError:
        print(f"[INFO] Installing missing dependency: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])

import openmeteo_requests
import requests_cache
import pandas as pd
import numpy as np
from retry_requests import retry
import time
from datetime import date, timedelta, datetime

# Setup client with cache and retry mechanism
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
client = openmeteo_requests.Client(session=retry_session)

In [2]:
# 2. Configuration Parameters
LAT = -7.736663290223948
LON = 109.64605763039752  # Jerukagung coordinates

# Detect environment (Local vs Kaggle)
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle')

if IS_KAGGLE:
    FOLDER = "/kaggle/working/forecast_open_meteo_jerukangung"
    print("[INFO] Running in Kaggle environment. Outputs will be saved to:", FOLDER)
else:
    FOLDER = "../forecast_open_meteo_jerukangung"
    print("[INFO] Running in Local environment. Outputs will be saved to:", FOLDER)

FILE_FINAL = "cuaca_gabungan_jerukagung.csv"

HOURLY_VARS = [
    "temperature_2m", "wet_bulb_temperature_2m", "relative_humidity_2m", 
    "dew_point_2m", "rain", "et0_fao_evapotranspiration",
    "wind_speed_10m", "wind_gusts_10m", "wind_direction_10m", 
    "surface_pressure", "pressure_msl", "cloud_cover_high", 
    "cloud_cover_mid", "cloud_cover_low", "cloud_cover", 
    "vapour_pressure_deficit", "sunshine_duration", "shortwave_radiation", 
    "direct_radiation", "diffuse_radiation", "direct_normal_irradiance"
]

[INFO] Running in Local environment. Outputs will be saved to: ../forecast_open_meteo_jerukangung


In [3]:
# 3. Fetching Helper Functions
def process_data(response, hourly_vars):
    hourly = response.Hourly()
    date_range = pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    )

    data_dict = {"datetime": date_range}
    for i, var_name in enumerate(hourly_vars):
        data_dict[var_name] = hourly.Variables(i).ValuesAsNumpy()

    df = pd.DataFrame(data=data_dict)
    df['datetime'] = df['datetime'].dt.tz_convert('Asia/Jakarta')
    df.insert(1, 'unixtime', (df['datetime'].astype('int64') // 10**9))
    return df

def fetch_weather(client, lat, lon, start_date, end_date, api_type="archive"):
    if api_type == "archive":
        url = "https://archive-api.open-meteo.com/v1/archive"
    else:
        url = "https://api.open-meteo.com/v1/forecast"
        
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": HOURLY_VARS,
        "timezone": "Asia/Bangkok"
    }

    if api_type == "archive":
        params["models"] = "era5"
        
    # Try ecmwf_ifs022 first for forecast, fallback to default (seamless ECMWF) if it fails
    models_to_try = ["ecmwf_ifs022", None] if api_type == "forecast" else [None]
    
    for model_name in models_to_try:
        current_params = params.copy()
        if model_name:
            current_params["models"] = model_name
            
        print(f"[INFO] Fetching {api_type.upper()} data (model={model_name}) from {start_date} to {end_date}...")
        
        for attempt in range(3):
            try:
                responses = client.weather_api(url, params=current_params)
                return process_data(responses[0], HOURLY_VARS)
            except Exception as e:
                print(f"[WARN] Attempt {attempt+1} failed with error: {e}.")
                if attempt < 2:
                    print("Retrying in 5 seconds...")
                    time.sleep(5)
                else:
                    print("Failed all retries for this configuration.")
                    
    print(f"[ERROR] Failed to fetch {api_type} data!")
    return None

In [4]:
# 4. Fetch Historical (ERA5) and Forecast (ECMWF IFS)
archive_start = "2023-01-01"
archive_end_date = date.today() - timedelta(days=8)
archive_end = archive_end_date.strftime("%Y-%m-%d")

forecast_start_date = archive_end_date + timedelta(days=1)
forecast_start = forecast_start_date.strftime("%Y-%m-%d")
forecast_end = (date.today() + timedelta(days=7)).strftime("%Y-%m-%d")

df_archive = fetch_weather(client, LAT, LON, archive_start, archive_end, "archive")
df_forecast = fetch_weather(client, LAT, LON, forecast_start, forecast_end, "forecast")

[INFO] Fetching ARCHIVE data (model=None) from 2023-01-01 to 2026-06-13...
[INFO] Fetching FORECAST data (model=ecmwf_ifs022) from 2026-06-14 to 2026-06-28...


[WARN] Attempt 1 failed with error: failed to request 'https://api.open-meteo.com/v1/forecast': {'error': True, 'reason': "Data corrupted at path ''. Cannot initialize MultiDomains from invalid String value ecmwf_ifs022."}.
Retrying in 5 seconds...


[WARN] Attempt 2 failed with error: failed to request 'https://api.open-meteo.com/v1/forecast': {'error': True, 'reason': "Data corrupted at path ''. Cannot initialize MultiDomains from invalid String value ecmwf_ifs022."}.
Retrying in 5 seconds...


[WARN] Attempt 3 failed with error: failed to request 'https://api.open-meteo.com/v1/forecast': {'error': True, 'reason': "Data corrupted at path ''. Cannot initialize MultiDomains from invalid String value ecmwf_ifs022."}.
Failed all retries for this configuration.
[INFO] Fetching FORECAST data (model=None) from 2026-06-14 to 2026-06-28...


In [5]:
# 5. Combine and Merge Data
df_list = []
if df_archive is not None:
    df_list.append(df_archive)
if df_forecast is not None:
    df_list.append(df_forecast)

if df_list:
    df_final = pd.concat(df_list, ignore_index=True)
    df_final = df_final.drop_duplicates(subset=['datetime'], keep='first')
    df_final = df_final.set_index('datetime')
    df_final = df_final.sort_index()
    print(f"[OK] Combined dataset created. Total rows: {len(df_final):,}")
else:
    raise ValueError("No data was fetched successfully!")

[OK] Combined dataset created. Total rows: 30,600


In [6]:
# 6. Validate Data and Export to CSV
if not os.path.exists(FOLDER):
    os.makedirs(FOLDER, exist_ok=True)
    
path_final = os.path.join(FOLDER, FILE_FINAL)

# Verify bounds
nan_count = df_final.isna().sum().sum()
print(f"Validation summary:")
print(f"- Total rows: {len(df_final):,}")
print(f"- Date range: {df_final.index.min()} to {df_final.index.max()}")
print(f"- Total missing values (NaN): {nan_count}")
print(f"- Monotonically increasing index: {df_final.index.is_monotonic_increasing}")

# Export
df_final.to_csv(path_final)
print(f"\n[SUCCESS] Exported final combined weather data to: {path_final}")

Validation summary:
- Total rows: 30,600
- Date range: 2023-01-01 00:00:00+07:00 to 2026-06-28 23:00:00+07:00
- Total missing values (NaN): 0
- Monotonically increasing index: True



[SUCCESS] Exported final combined weather data to: ../forecast_open_meteo_jerukangung\cuaca_gabungan_jerukagung.csv
